# Sensitivity Analysis & Experimental Validation

This notebook elaborates on the experimental validation setup to demonstrate the exact sensitivity sweep curves where boundary decisions transition from `WARNING` to `CRITICAL` (emergency halt).

We utilize the `DynamicSafetyCalculator` to sweep across multiple velocities and latencies.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add backend to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'backend')))

from app.safety.dynamic_zone_calculator import DynamicSafetyCalculator

## 1. Sensitivity Sweep: Robot Speed vs Required Separation

We sweep the robot speed from $0.5$ m/s to $3.0$ m/s while keeping human speed constant at $1.2$ m/s, to observe how the `CRITICAL` boundaries shift.

In [ ]:
robot_speeds = np.linspace(0.5, 3.0, 50)
human_speed = 1.2
reaction_time = 0.25
decel = 1.2
actual_distance_fixed = 3.0 # Operator is 3.0m away

s_reqs = []
risk_levels = []

for rs in robot_speeds:
    req_sep, _ = DynamicSafetyCalculator.calculate_required_separation(
        robot_speed=rs,
        human_speed=human_speed,
        reaction_time=reaction_time,
        decel=decel,
        pos_uncertainty=0.15,
        sensor_latency=0.12
    )
    s_reqs.append(req_sep)
    risk, _ = DynamicSafetyCalculator.classify_risk(actual_distance_fixed, req_sep)
    risk_levels.append(risk)

colors = {'SAFE': 'green', 'WARNING': 'yellow', 'HIGH_RISK': 'orange', 'CRITICAL': 'red'}
mapped_colors = [colors[r] for r in risk_levels]

plt.figure(figsize=(10, 6))
plt.scatter(robot_speeds, s_reqs, c=mapped_colors, label='Dynamic S_req')
plt.axhline(y=actual_distance_fixed, color='blue', linestyle='--', label='Actual Distance (3.0m)')
plt.title("Boundary Transition: Robot Speed vs Required Separation")
plt.xlabel("Robot Speed (m/s)")
plt.ylabel("Required Separation (m)")
plt.legend()
plt.grid(True)
plt.show()

## 2. Decision Surface: Human Speed & Robot Speed

Let's visualize the 2D surface of the required separation.

In [ ]:
human_speeds = np.linspace(0.0, 2.5, 30)
X, Y = np.meshgrid(robot_speeds, human_speeds)
Z = np.zeros_like(X)

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        req_sep, _ = DynamicSafetyCalculator.calculate_required_separation(
            robot_speed=X[i,j],
            human_speed=Y[i,j],
            reaction_time=reaction_time,
            decel=decel
        )
        Z[i,j] = req_sep

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='viridis')
ax.set_xlabel('Robot Speed (m/s)')
ax.set_ylabel('Human Speed (m/s)')
ax.set_zlabel('Required Separation (m)')
fig.colorbar(surf, shrink=0.5, aspect=5)
plt.title("Decision Surface for Dynamic Separation")
plt.show()